# 3 · Cross-Validation on FABRIC

Compare four BB84 results for the **same scenario**, all executed on the FABRIC slice:

| Backend | Where it runs | What it is |
|---------|---------------|------------|
| **QFabric (measured)** | BMv2 data plane (from `02_run_experiment`) | the real emulation over FABRIC |
| **QFabric-sim** | switch node (`.venv-qsim`) | pure-Python model, no traffic |
| **SeQUeNCe** | alice node (`.venv-seq`, Python 3.12) | SeQUeNCe 1.0 native engine |
| **NetSquid** | bob node (`.venv-nsq`) | NetSquid native engine |

Everything runs on FABRIC infrastructure — the simulators execute on the slice nodes
via `fablib`, not in this kernel. Uninstalled/failed backends are reported **SKIPPED**.

**Prerequisite:** `01_setup_slice` + `02_run_experiment` completed.

### At a glance
- **Purpose:** compare the *measured* FABRIC QBER against QFabric-sim, SeQUeNCe, and NetSquid for one scenario — the core scientific check.
- **Inputs:** `SLICE_NAME`, `SCENARIO`, and (Step 1) `NETSQUID_USER`/`NETSQUID_PASS` to install NetSquid on the bob node.
- **Outputs:** a backend table + pairwise AGREE/DIFFER report, saved to `results/cross_validation.json` (consumed by notebook 4).
- **Runs on / runtime:** FABRIC JupyterHub (simulators run on the slice nodes); Step 1 env build is **~10 min one-time**, the comparison **~3–5 min**.
- **If something fails:** a backend that errors/has no key shows **SKIPPED** (with the reason) — never a false pass. Skip Step 1 on later runs once the `.venv-*` dirs exist.

## Configuration
Match `SLICE_NAME` / `SCENARIO` to notebooks 1–2. For NetSquid, set your netsquid.org
credentials in the environment before running (used to install NetSquid on the bob node):
`os.environ['NETSQUID_USER']=...`, `os.environ['NETSQUID_PASS']=...`

In [1]:
SLICE_NAME = 'qfabric-bb84-2'
SCENARIO   = 'validation/scenarios/fabric_1km.yml'   # same scenario as notebook 2

In [2]:
import os, sys
from pathlib import Path

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'qne').is_dir())
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
import deploy_fabric as df
from validation.compare import compare_results, print_backend_summary

fablib = df.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/Users/kthare10/work/fabric_config/id_token.json
Project ID,4604cab7-41ff-4c1a-a935-0ca6f20cceeb
Bastion Host,bastion.fabric-testbed.net
Bastion Username,kthare10_0011904101
Bastion Private Key File,/Users/kthare10/.ssh/bastion-prod-2
Slice Public Key File,/Users/kthare10/.ssh/id_rsa.pub
Slice Private Key File,/Users/kthare10/.ssh/id_rsa


User: kthare10@email.unc.edu bastion key is valid!
Configuration is valid


ID,9e082748-bdc9-4c91-b8b7-4b3753bf9f9c
Name,qfabric-bb84-2
Lease Expiration (UTC),2026-10-04 02:31:48 +0000
Lease Start (UTC),2026-07-22 02:31:48 +0000
Project ID,4604cab7-41ff-4c1a-a935-0ca6f20cceeb
State,StableOK
Email,kthare10@email.unc.edu
UserId,43b7271b-90eb-45f6-833a-e51cf13bbc68


ID,9e082748-bdc9-4c91-b8b7-4b3753bf9f9c
Name,qfabric-bb84-2
Lease Expiration (UTC),2026-10-04 02:31:48 +0000
Lease Start (UTC),2026-07-22 02:31:48 +0000
Project ID,4604cab7-41ff-4c1a-a935-0ca6f20cceeb
State,StableOK
Email,kthare10@email.unc.edu
UserId,43b7271b-90eb-45f6-833a-e51cf13bbc68


## Step 1 — Build simulator envs on the nodes (one-time)
Installs SeQUeNCe 1.0 (Python 3.12 via deadsnakes) on alice, NetSquid on bob, and a
QFabric-sim venv on the switch. Idempotent and takes several minutes the first time.
Skip on later runs once the `.venv-*` dirs exist.

In [3]:
import os, getpass

# NetSquid creds (needed only to install NetSquid on the bob node).
# Preferred: export NETSQUID_USER / NETSQUID_PASS *before* launching Jupyter so
# the kernel inherits them. If they're missing (or still the '...' placeholder),
# prompt now — this keeps the password OUT of the notebook source and outputs.
if os.environ.get('NETSQUID_USER', '...') in ('', '...'):
    os.environ['NETSQUID_USER'] = input('netsquid.org username: ').strip()
if os.environ.get('NETSQUID_PASS', '...') in ('', '...'):
    os.environ['NETSQUID_PASS'] = getpass.getpass('netsquid.org password: ')

assert os.environ['NETSQUID_USER'] and os.environ['NETSQUID_PASS'], \
    'NetSquid credentials are empty — set them and re-run this cell.'
print(f"NetSquid user: {os.environ['NETSQUID_USER']}  (password hidden, "
      f"{len(os.environ['NETSQUID_PASS'])} chars)")

df.setup_sim_envs(slice_obj)

NetSquid user: kthare10  (password hidden, 12 chars)

=== Setting up simulator envs on FABRIC nodes (one-time, several min) ===
  [switch] QFabric-sim (.venv-qsim)...
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  build-essential bzip2 cpp cpp-11 dpkg-dev fakeroot fontconfig-config
  fonts-dejavu-core g++ g++-11 gcc gcc-11 gcc-11-base javascript-common
  libalgorithm-diff-perl libalgorithm-diff-xs-perl libalgorithm-merge-perl
  libasan6 libatomic1 libc-dev-bin libc-devtools libc6-dev libcc1-0
  libcrypt-dev libdeflate0 libdpkg-perl libexpat1-dev libfakeroot
  libfile-fcntllock-perl libfontconfig1 libgcc-11-dev libgd3 libgomp1 libisl23
  libitm1 libjbig0 libjpeg-turbo8 libjpeg8 libjs-jquery libjs-sphinxdoc
  libjs-underscore liblsan0 libmpc3 libnsl-dev libpython3-dev
  libpython3.10-dev libquadmath0 libstdc++-11-dev libtiff5 libtirpc-dev
  libtsan0 libubsan1 libwebp7 libxpm4 linux-libc-dev lto-disa

## Step 2 — Run the cross-validation on FABRIC

In [ ]:
import json
from validation.compare import scenario_from_fabric_result

results = df.run_cross_validation_on_fabric(slice_obj, SCENARIO)

# Expected intrinsic QBER for this scenario: depolarizing misalignment ~ (1 - F)/2.
try:
    fid = scenario_from_fabric_result(PROJECT_DIR/'results'/'fabric_bob_results.json').polarization_fidelity
    print(f'Expected QBER for polarization fidelity {fid}: ~{(1-fid)/2:.4f}\n')
except Exception:
    pass

ok = print_backend_summary(results)
comp = compare_results(ok)

print('\n--- Pairwise agreement (dQBER vs 2-sigma statistical tolerance) ---')
for c in comp['comparisons']:
    verdict = 'AGREE' if c['passed'] else 'DIFFER'
    print(f"  [{verdict}] {c['platform_a']:12s} vs {c['platform_b']:12s}  "
          f"QBER {c['qber_a']:.4f} / {c['qber_b']:.4f}  "
          f"dQBER={c['delta_qber']:.4f}  tol={c['tolerance']:.4f}")

n_pass = sum(c['passed'] for c in comp['comparisons']); n_tot = len(comp['comparisons'])
if n_tot == 0:
    print('\nINCONCLUSIVE - need >=2 backends with data (see SKIPPED above).')
else:
    print(f'\n{n_pass}/{n_tot} backend pairs agree within 2 sigma.')

# Persist for notebook 04's figures.
out = PROJECT_DIR / 'results' / 'cross_validation.json'
out.write_text(json.dumps([r.to_payload() for r in results], indent=2))
print(f'Saved cross-validation results -> {out}')

### How to read this

Each backend reports the **QBER** (quantum bit error rate) for the *same* scenario:

| Backend | Source |
|---------|--------|
| `qfabric` | measured on the FABRIC **BMv2 data plane** (the real emulation) |
| `qfabric_sim` | QFabric's pure-Python model (switch node) |
| `sequence` | **SeQUeNCe** simulator (alice node) |
| `netsquid` | **NetSquid** simulator (bob node) |

For polarization fidelity *F*, the expected intrinsic QBER is **(1 − F)/2** (≈ 1% at F = 0.98).

For every pair we show **ΔQBER** and a **2σ tolerance** — the statistical noise expected from
finite key sizes, `2·√(p(1−p)(1/Nₐ + 1/N_b))`:

- **AGREE** (ΔQBER < tol): the two platforms give statistically indistinguishable QBER.
- **DIFFER** (ΔQBER ≥ tol): the gap exceeds statistical noise. QFabric-vs-simulator would flag
  an emulation problem; **simulator-vs-simulator usually reflects a genuine difference in their
  physics models** (e.g. how fidelity maps to QBER) — a normal, reportable cross-validation result.

**Headline:** does QFabric *measured on FABRIC* agree with the established simulators? Look at the `qfabric` rows.

## Step 3 — Summary table

In [ ]:
import pandas as pd
rows = [{'platform': r.platform, 'QBER': round(r.qber, 4), 'sifted_bits': r.sifted_bits,
         'secure_key_rate': round(r.secure_key_rate, 4),
         'status': 'ok' if (not r.extra.get('error') and r.sifted_bits > 0) else 'skipped'}
        for r in results]
pd.DataFrame(rows)

---
**Next:** `04_analysis` for the FABRIC pipeline plots and loss-budget breakdown.